# LangChain Document এবং Document Loaders

এই নোটবুকে আমরা শিখব:

| বিষয় | বিবরণ |
|---|---|
| `Document` structure | `page_content` এবং `metadata` কী |
| `PyPDFLoader` | PDF ফাইল লোড করার পদ্ধতি |
| `CSVLoader` | CSV ফাইল লোড করার পদ্ধতি |
| `WebBaseLoader` | ওয়েব পেজ থেকে ডেটা লোড করার পদ্ধতি |
| `DirectoryLoader` | একটি ফোল্ডারের সব ফাইল একসাথে লোড করার পদ্ধতি |

**Data Directory Structure:**
```
RAG/data/
├── pdf/
│   ├── langchain_intro.pdf
│   └── rag_overview.pdf
├── text_files/
│   ├── artificial_intelligence.txt
│   ├── machine_learning.txt
│   └── natural_language_processing.txt
└── csv/
    └── students.csv
```

### Step 1 — Environment Variables লোড করা

**What:** `.env` ফাইল থেকে API keys লোড করা হচ্ছে।

**Why:** LangChain-এর সব provider class গুলো API key দরকার করে। এখানে একবার লোড করলে পুরো নোটবুকে ব্যবহার করা যাবে।

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

ANTHROPIC_API_KEY: True


---
## 1. LangChain `Document` Structure

### Document কী?

LangChain-এ **`Document`** হল একটি মৌলিক ডেটা স্ট্রাকচার যা যেকোনো text content এবং তার সাথে সম্পর্কিত metadata ধারণ করে। RAG (Retrieval Augmented Generation) pipeline-এর সব কিছু এই `Document` অবজেক্টকে কেন্দ্র করে।

### দুটি মূল Field:

| Field | Type | বিবরণ |
|---|---|---|
| `page_content` | `str` | মূল text content — যা LLM পড়বে |
| `metadata` | `dict` | Content সম্পর্কে তথ্য — source, page নম্বর, author ইত্যাদি |

**কেন `metadata` গুরুত্বপূর্ণ?**
- কোথা থেকে ডেটা এসেছে তা ট্র্যাক করতে (source attribution)
- Filtering করতে — শুধু নির্দিষ্ট source বা page থেকে retrieve করতে
- User-কে citation দেখাতে

In [2]:
from langchain_core.documents import Document

# একটি Document manually তৈরি করা
doc = Document(
    page_content="LangChain হল একটি open-source framework যা LLM-powered applications তৈরি করতে সাহায্য করে।",
    metadata={
        "source": "manual_example",
        "author": "Harrison Chase",
        "year": 2022,
        "language": "Bangla",
    }
)

print("=== Document Structure ===")
print(f"page_content: {doc.page_content}")
print(f"metadata:     {doc.metadata}")
print(f"\nType: {type(doc)}")

=== Document Structure ===
page_content: LangChain হল একটি open-source framework যা LLM-powered applications তৈরি করতে সাহায্য করে।
metadata:     {'source': 'manual_example', 'author': 'Harrison Chase', 'year': 2022, 'language': 'Bangla'}

Type: <class 'langchain_core.documents.base.Document'>


In [3]:
# একাধিক Document তৈরি করে list এ রাখা — Document Loaders এভাবেই রিটার্ন করে
docs = [
    Document(
        page_content="Machine Learning হল AI-এর একটি subset যেখানে computer data থেকে শেখে।",
        metadata={"source": "ml_book.pdf", "page": 1, "chapter": "Introduction"}
    ),
    Document(
        page_content="Deep Learning neural network ব্যবহার করে complex pattern recognize করে।",
        metadata={"source": "ml_book.pdf", "page": 2, "chapter": "Deep Learning"}
    ),
    Document(
        page_content="Natural Language Processing মানুষের ভাষা বুঝতে সাহায্য করে।",
        metadata={"source": "ml_book.pdf", "page": 3, "chapter": "NLP"}
    ),
]

print(f"Total Documents: {len(docs)}\n")
for i, d in enumerate(docs):
    print(f"Document {i+1}:")
    print(f"  content  : {d.page_content[:60]}...")
    print(f"  metadata : {d.metadata}")
    print()

Total Documents: 3

Document 1:
  content  : Machine Learning হল AI-এর একটি subset যেখানে computer data থ...
  metadata : {'source': 'ml_book.pdf', 'page': 1, 'chapter': 'Introduction'}

Document 2:
  content  : Deep Learning neural network ব্যবহার করে complex pattern rec...
  metadata : {'source': 'ml_book.pdf', 'page': 2, 'chapter': 'Deep Learning'}

Document 3:
  content  : Natural Language Processing মানুষের ভাষা বুঝতে সাহায্য করে।...
  metadata : {'source': 'ml_book.pdf', 'page': 3, 'chapter': 'NLP'}



In [4]:
# Metadata access করার পদ্ধতি
first_doc = docs[0]

print("Source:", first_doc.metadata["source"])
print("Page:",   first_doc.metadata["page"])
print("Chapter:", first_doc.metadata.get("chapter", "Unknown"))

# Metadata filter করা — শুধু page 2 বা তার বেশি
filtered = [d for d in docs if d.metadata["page"] >= 2]
print(f"\nPage >= 2 filtered docs: {len(filtered)}")

Source: ml_book.pdf
Page: 1
Chapter: Introduction

Page >= 2 filtered docs: 2


---
## 2. Document Loaders কী?

**Document Loaders** হল LangChain-এর এমন ক্লাস যা বিভিন্ন source থেকে data পড়ে `Document` object-এর list হিসেবে রিটার্ন করে।

### Loader-এর Common Interface:

```python
loader = SomeLoader("path/or/url")
docs = loader.load()          # সব document একসাথে লোড করে
# অথবা
docs = loader.lazy_load()     # একটা একটা করে yield করে (memory efficient)
```

### Available Document Loaders:

| Loader | Source | Package |
|---|---|---|
| `PyPDFLoader` | PDF files | `langchain_community` |
| `CSVLoader` | CSV files | `langchain_community` |
| `WebBaseLoader` | Web pages | `langchain_community` |
| `DirectoryLoader` | Folder of files | `langchain_community` |
| `TextLoader` | Plain text files | `langchain_community` |
| `UnstructuredWordDocumentLoader` | `.docx` files | `langchain_community` |
| `JSONLoader` | JSON files | `langchain_community` |

---
## 3. PyPDFLoader — PDF থেকে Document লোড করা

### কীভাবে কাজ করে?

**`PyPDFLoader`** একটি PDF ফাইলের প্রতিটি page-কে আলাদা `Document` হিসেবে লোড করে।

- `page_content` → সেই page-এর text content
- `metadata` → `{"source": "file.pdf", "page": 0}` (page 0-indexed)

**Backend:** `pypdf` library ব্যবহার করে (আমরা আগেই `uv add pypdf` করেছি)

In [5]:
from langchain_community.document_loaders import PyPDFLoader

# PDF file load করা
pdf_path = "../data/pdf/langchain_intro.pdf"
loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load()

print(f"PDF: {pdf_path}")
print(f"Total pages loaded: {len(pdf_docs)}")
print()

for i, doc in enumerate(pdf_docs):
    print(f"--- Page {i + 1} ---")
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print()

C:\Users\Nibras\AppData\Local\Temp\ipykernel_21244\3636090521.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF: ../data/pdf/langchain_intro.pdf
Total pages loaded: 1

--- Page 1 ---
Content preview: Introduction to LangChain
What is LangChain?
LangChain is an open-source framework for building LLM-powered applications. It provides standardized
interfaces for models, prompts, chains, memory, agent...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-07-02T16:54:31+00:00', 'source': '../data/pdf/langchain_intro.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}



In [6]:
# দ্বিতীয় PDF লোড করা
pdf_path2 = "../data/pdf/rag_overview.pdf"
loader2 = PyPDFLoader(pdf_path2)
rag_docs = loader2.load()

print(f"PDF: {pdf_path2}")
print(f"Total pages loaded: {len(rag_docs)}\n")

print("Full content of page 1:")
print(rag_docs[0].page_content)

PDF: ../data/pdf/rag_overview.pdf
Total pages loaded: 1

Full content of page 1:
Retrieval Augmented Generation (RAG)
What is RAG?
RAG enhances LLM responses by retrieving relevant documents before generating answers. It combines retrieval
systems with generative AI for accurate, grounded outputs.
RAG Pipeline Steps
1. Document Loading: Ingest documents from PDFs, web, databases, files.
2. Text Splitting: Chunk documents into smaller, overlapping pieces.
3. Embedding: Convert chunks to dense vector representations.
4. Vector Store: Index vectors in a database (FAISS, Chroma, Pinecone).
5. Retrieval: Semantic search to find top-k relevant chunks for a query.
6. Generation: LLM reads retrieved context and generates the final answer.
Benefits of RAG
Reduces hallucinations by grounding answers in retrieved evidence.
Supports knowledge cutoff override - use real-time or proprietary data.
More cost-effective than fine-tuning for domain knowledge updates.
Enables source citation and auditabil

In [7]:
# Anthropic দিয়ে PDF content সম্পর্কে প্রশ্ন করা
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

pdf_text = "\n".join([d.page_content for d in pdf_docs])

question = (
    "নিচের text পড়ে বাংলায় ২-৩ বাক্যে সারসংক্ষেপ করো:\n\n"
    + pdf_text[:1000]
)

response = llm.invoke(question)
print("PDF Summary (Bangla):")
print(response.content)

PDF Summary (Bangla):
# সারসংক্ষেপ

**LangChain** একটি ওপেন-সোর্স ফ্রেমওয়ার্ক যা বড় ভাষা মডেল (LLM) দিয়ে চালিত অ্যাপ্লিকেশন তৈরির জন্য ব্যবহৃত হয়। এটি মডেল, প্রম্পট, চেইন, মেমরি, এজেন্ট, টুলস এবং ভেক্টর স্টোরের মতো বিভিন্ন প্রয়োজনীয় উপাদানের জন্য স্ট্যান্ডার্ড ইন্টারফেস প্রদান করে। LangChain Expression Language (LCEL) ব্যবহার করে পাইপ অপারেটর (|) এর মাধ্যমে চেইনগুলিকে সহজভাবে সংযুক্ত করা যায়, যা স্ট্রিমিং, সমান্তরাল সম্পাদন এবং অ্যাসিঙ্ক সাপোর্টের সুবিধা দেয়।


---
## 4. CSVLoader — CSV ফাইল থেকে Document লোড করা

### কীভাবে কাজ করে?

**`CSVLoader`** একটি CSV ফাইলের প্রতিটি **row**-কে আলাদা `Document` হিসেবে লোড করে।

- `page_content` → সব column-এর key-value pair (formatted as text)
- `metadata` → `{"source": "file.csv", "row": 0}` (row 0-indexed)

### Optional Parameters:
- `source_column`: কোন column-কে metadata `source` হিসেবে ব্যবহার করা হবে
- `csv_args`: Python `csv.reader`-এর arguments (delimiter, quotechar ইত্যাদি)

In [8]:
from langchain_community.document_loaders.csv_loader import CSVLoader

csv_path = "../data/csv/students.csv"
loader = CSVLoader(file_path=csv_path)
csv_docs = loader.load()

print(f"CSV: {csv_path}")
print(f"Total rows loaded: {len(csv_docs)}")
print()

for i, doc in enumerate(csv_docs[:3]):
    print(f"--- Row {i + 1} ---")
    print(f"Content:\n{doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

CSV: ../data/csv/students.csv
Total rows loaded: 10

--- Row 1 ---
Content:
name: Alice Johnson
age: 22
subject: Machine Learning
grade: A
city: New York
Metadata: {'source': '../data/csv/students.csv', 'row': 0}

--- Row 2 ---
Content:
name: Bob Smith
age: 24
subject: Deep Learning
grade: B+
city: San Francisco
Metadata: {'source': '../data/csv/students.csv', 'row': 1}

--- Row 3 ---
Content:
name: Carol White
age: 21
subject: Natural Language Processing
grade: A-
city: Boston
Metadata: {'source': '../data/csv/students.csv', 'row': 2}



In [9]:
# source_column ব্যবহার করা — metadata source হিসেবে name column
loader_named = CSVLoader(
    file_path=csv_path,
    source_column="name",
)
named_docs = loader_named.load()

print("source_column=name ব্যবহার করলে metadata:")
for doc in named_docs[:3]:
    print(f"  source: {doc.metadata['source']}, row: {doc.metadata['row']}")

source_column=name ব্যবহার করলে metadata:
  source: Alice Johnson, row: 0
  source: Bob Smith, row: 1
  source: Carol White, row: 2


In [10]:
# LLM দিয়ে CSV data বিশ্লেষণ করা
all_csv_text = "\n\n".join([d.page_content for d in csv_docs])

question = (
    "নিচে একটি students dataset আছে। বাংলায় বলো:\n"
    "১. কতজন ছাত্র আছে?\n"
    "২. সর্বোচ্চ grade (A+) কে পেয়েছে?\n"
    "৩. কোন শহরে সবচেয়ে বেশি ছাত্র আছে?\n\n"
    + all_csv_text
)

response = llm.invoke(question)
print("CSV Analysis (Bangla):")
print(response.content)

CSV Analysis (Bangla):
# Dataset বিশ্লেষণ

## ১. কতজন ছাত্র আছে?
**মোট ১০ জন ছাত্র আছে।**

## ২. সর্বোচ্চ grade (A+) কে পেয়েছে?
**দুইজন ছাত্র A+ grade পেয়েছে:**
- **Eva Martinez** (Reinforcement Learning)
- **James Taylor** (Mathematics for AI)

## ৩. কোন শহরে সবচেয়ে বেশি ছাত্র আছে?
**কোন শহরেই একাধিক ছাত্র নেই।** প্রতিটি শহরে মাত্র ১ জন করে ছাত্র আছে:
- New York, San Francisco, Boston, Chicago, Austin, Seattle, Los Angeles, Denver, Miami, Portland - প্রতিটিতে ১ জন

তাই সব শহর সমানভাবে প্রতিনিধিত্ব পাচ্ছে।


---
## 5. WebBaseLoader — ওয়েব পেজ থেকে Document লোড করা

### কীভাবে কাজ করে?

**`WebBaseLoader`** একটি URL থেকে HTML পেজ fetch করে, BeautifulSoup দিয়ে parse করে, এবং clean text হিসেবে `Document` রিটার্ন করে।

- `page_content` → পেজের visible text content (HTML tags ছাড়া)
- `metadata` → `{"source": "url", "title": "page title", "description": "...", "language": "en"}`

### একাধিক URL একসাথে:
```python
loader = WebBaseLoader(["url1", "url2", "url3"])
docs = loader.load()  # সব URL থেকে লোড করবে
```

**Dependency:** `beautifulsoup4` (langchain_community এর সাথে installed হয়)

In [12]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://en.wikipedia.org/wiki/LangChain"
loader = WebBaseLoader(url)
web_docs = loader.load()

print(f"URL: {url}")
print(f"Total documents loaded: {len(web_docs)}")
print()
print("Metadata:")
for key, value in web_docs[0].metadata.items():
    print(f"  {key}: {str(value)[:100]}")
print()
print(f"Content length: {len(web_docs[0].page_content)} characters")
print(f"\nContent preview (first 500 chars):")
print(web_docs[0].page_content[:500])

URL: https://en.wikipedia.org/wiki/LangChain
Total documents loaded: 1

Metadata:
  source: https://en.wikipedia.org/wiki/LangChain
  title: LangChain - Wikipedia
  language: en

Content length: 18232 characters

Content preview (first 500 chars):




LangChain - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in










In [13]:
# একাধিক URL একসাথে লোড করা
urls = [
    "https://en.wikipedia.org/wiki/Large_language_model",
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
]

multi_loader = WebBaseLoader(urls)
multi_docs = multi_loader.load()

print(f"Total URLs: {len(urls)}")
print(f"Total documents: {len(multi_docs)}\n")

for doc in multi_docs:
    title = doc.metadata.get("title", "No title")
    source = doc.metadata.get("source", "No source")
    print(f"Title: {title}")
    print(f"Source: {source}")
    print(f"Content length: {len(doc.page_content)} chars")
    print()

Total URLs: 2
Total documents: 2

Title: Large language model - Wikipedia
Source: https://en.wikipedia.org/wiki/Large_language_model
Content length: 114642 chars

Title: Retrieval-augmented generation - Wikipedia
Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Content length: 24365 chars



In [14]:
# WebBaseLoader এর content দিয়ে LLM-কে প্রশ্ন করা
web_content = web_docs[0].page_content[:2000]

question = (
    "নিচের text পড়ে বাংলায় বলো: Large Language Model কী এবং এটি কীভাবে কাজ করে?\n"
    "সংক্ষেপে ৩-৪ বাক্যে উত্তর দাও।\n\nText:\n"
    + web_content
)

response = llm.invoke(question)
print("Wikipedia থেকে পড়া content এর উপর LLM এর উত্তর:")
print(response.content)

Wikipedia থেকে পড়া content এর উপর LLM এর উত্তর:
# Large Language Model (LLM) কী এবং কীভাবে কাজ করে?

প্রদত্ত টেক্সটে সরাসরি LLM এর সংজ্ঞা নেই, তবে এটি একটি **LangChain ফ্রেমওয়ার্ক সম্পর্কিত নথি** যা LLM গুলিকে অ্যাপ্লিকেশনে সংযুক্ত করতে সাহায্য করে। 

সাধারণভাবে বলতে গেলে: **Large Language Model (LLM) হল একটি কৃত্রিম বুদ্ধিমত্তা মডেল যা বিশাল পরিমাণ টেক্সট ডেটা থেকে শিখে মানুষের মতো প্রাকৃতিক ভাষা বোঝে এবং তৈরি করে।** এটি শব্দের সম্ভাব্যতার উপর ভিত্তি করে পরবর্তী শব্দ পূর্বাভাস দিয়ে কাজ করে এবং পুনরাবৃত্তির মাধ্যমে সম্পূর্ণ উত্তর বা পাঠ্য তৈরি করে।


---
## 6. DirectoryLoader — ফোল্ডারের সব ফাইল লোড করা

### কীভাবে কাজ করে?

**`DirectoryLoader`** একটি directory-তে থাকা সব ফাইল (নির্দিষ্ট pattern অনুযায়ী) স্বয়ংক্রিয়ভাবে লোড করে।

```python
DirectoryLoader(
    path="./data",           # directory path
    glob="**/*.txt",         # কোন ফাইল pattern লোড করতে হবে
    loader_cls=TextLoader,   # কোন Loader class ব্যবহার করতে হবে
    show_progress=True,      # progress bar দেখাবে
    use_multithreading=True, # parallel লোড করবে
)
```

### Glob Patterns:

| Pattern | মানে |
|---|---|
| `**/*.txt` | সব sub-folder-এ `.txt` ফাইল |
| `*.pdf` | শুধু root directory-তে `.pdf` ফাইল |
| `**/*.{txt,md}` | `.txt` বা `.md` ফাইল (সব folder-এ) |

In [15]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_path = "../data/text_files"
loader = DirectoryLoader(
    path=dir_path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True,
)
dir_docs = loader.load()

print(f"\nDirectory: {dir_path}")
print(f"Total documents loaded: {len(dir_docs)}")
print()

for doc in dir_docs:
    filename = os.path.basename(doc.metadata["source"])
    print(f"File: {filename}")
    print(f"  Content length: {len(doc.page_content)} chars")
    print(f"  Preview: {doc.page_content[:100].strip()}...")
    print()

100%|██████████| 3/3 [00:00<00:00, 2173.21it/s]


Directory: ../data/text_files
Total documents loaded: 3

File: artificial_intelligence.txt
  Content length: 2182 chars
  Preview: Artificial Intelligence: An Overview

Artificial Intelligence (AI) is a branch of computer science t...

File: machine_learning.txt
  Content length: 2540 chars
  Preview: Machine Learning: A Comprehensive Guide

Machine Learning (ML) is a field of artificial intelligence...

File: natural_language_processing.txt
  Content length: 2810 chars
  Preview: Natural Language Processing: Understanding Human Language

Natural Language Processing (NLP) is a br...



In [16]:
# UTF-8 encoding explicitly set করা
loader_utf8 = DirectoryLoader(
    path=dir_path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False,
)
utf8_docs = loader_utf8.load()

print(f"UTF-8 encoding দিয়ে লোড হওয়া docs: {len(utf8_docs)}")
print("\nMetadata of each document:")
for doc in utf8_docs:
    print(f"  {doc.metadata}")

UTF-8 encoding দিয়ে লোড হওয়া docs: 3

Metadata of each document:
  {'source': '..\\data\\text_files\\artificial_intelligence.txt'}
  {'source': '..\\data\\text_files\\machine_learning.txt'}
  {'source': '..\\data\\text_files\\natural_language_processing.txt'}


In [17]:
# PDF directory থেকেও DirectoryLoader দিয়ে লোড করা
pdf_dir_loader = DirectoryLoader(
    path="../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
)
pdf_dir_docs = pdf_dir_loader.load()

print(f"\nPDF directory থেকে লোড: {len(pdf_dir_docs)} documents")
for doc in pdf_dir_docs:
    print(f"  Source: {os.path.basename(doc.metadata['source'])}, Page: {doc.metadata.get('page', 'N/A')}")

100%|██████████| 2/2 [00:00<00:00, 281.43it/s]


PDF directory থেকে লোড: 2 documents
  Source: langchain_intro.pdf, Page: 0
  Source: rag_overview.pdf, Page: 0


In [18]:
# DirectoryLoader থেকে পাওয়া সব text দিয়ে LLM-কে প্রশ্ন করা
all_text = "\n\n".join([d.page_content for d in dir_docs])

question = (
    "নিচে তিনটি আলাদা বিষয়ের উপর text দেওয়া আছে।\n"
    "বাংলায় প্রতিটি বিষয়ের নাম ও একটি করে মূল বিষয় বলো:\n\n"
    + all_text[:3000]
)

response = llm.invoke(question)
print("Directory থেকে লোড হওয়া সব file এর summary:")
print(response.content)

Directory থেকে লোড হওয়া সব file এর summary:
# তিনটি বিষয়ের নাম ও মূল বিষয়:

## ১. **কৃত্রিম বুদ্ধিমত্তা (Artificial Intelligence)**
**মূল বিষয়:** কৃত্রিম বুদ্ধিমত্তা হল কম্পিউটার বিজ্ঞানের একটি শাখা যা মানুষের মতো বুদ্ধিমত্তা সম্পন্ন যন্ত্র তৈরি করার লক্ষ্যে কাজ করে, যা প্রাকৃতিক ভাষা বোঝা, প্যাটার্ন চেনা এবং সিদ্ধান্ত গ্রহণ করতে পারে।

## ২. **মেশিন লার্নিং (Machine Learning)**
**মূল বিষয়:** মেশিন লার্নিং হল কৃত্রিম বুদ্ধিমত্তার একটি উপশাখা যা কম্পিউটারকে স্পষ্ট নির্দেশ ছাড়াই ডেটা থেকে শিখে এবং উন্নত হতে সক্ষম করে।

## ৩. **গভীর শিক্ষা (Deep Learning)**
**মূল বিষয়:** গভীর শিক্ষা বহু-স্তরীয় নিউরাল নেটওয়ার্ক ব্যবহার করে জটিল ডেটা প্রক্রিয়া করে এবং ছবি চেনা, বক্তৃতা প্রক্রিয়াকরণ ও ভাষা বোঝার মতো অত্যাধুনিক প্রযুক্তিতে শক্তি সরবরাহ করে।


---
## 7. সব Loader একসাথে তুলনা

In [19]:
print("=" * 60)
print("Document Loader তুলনা")
print("=" * 60)

summary = [
    ("PyPDFLoader", len(pdf_docs), "PDF pages", pdf_docs[0].metadata),
    ("CSVLoader", len(csv_docs), "CSV rows", csv_docs[0].metadata),
    ("WebBaseLoader", len(web_docs), "web pages", {k: str(v)[:40] for k, v in web_docs[0].metadata.items()}),
    ("DirectoryLoader", len(dir_docs), "txt files", dir_docs[0].metadata),
]

for loader_name, doc_count, unit, sample_meta in summary:
    print(f"\n{loader_name}:")
    print(f"  Loaded: {doc_count} documents ({unit})")
    print(f"  Sample metadata: {sample_meta}")

Document Loader তুলনা

PyPDFLoader:
  Loaded: 1 documents (PDF pages)
  Sample metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-07-02T16:54:31+00:00', 'source': '../data/pdf/langchain_intro.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}

CSVLoader:
  Loaded: 10 documents (CSV rows)
  Sample metadata: {'source': '../data/csv/students.csv', 'row': 0}

WebBaseLoader:
  Loaded: 1 documents (web pages)
  Sample metadata: {'source': 'https://en.wikipedia.org/wiki/LangChain', 'title': 'LangChain - Wikipedia', 'language': 'en'}

DirectoryLoader:
  Loaded: 3 documents (txt files)
  Sample metadata: {'source': '..\\data\\text_files\\artificial_intelligence.txt'}


---
## সারসংক্ষেপ

### এই নোটবুকে যা শিখলাম:

| Concept | মূল কথা |
|---|---|
| `Document` | LangChain-এর মূল ডেটা স্ট্রাকচার — `page_content` + `metadata` |
| `page_content` | যে text LLM পড়বে (str) |
| `metadata` | Source, page, author ইত্যাদি তথ্য (dict) |
| `PyPDFLoader` | PDF → প্রতি page = একটি Document |
| `CSVLoader` | CSV → প্রতি row = একটি Document |
| `WebBaseLoader` | URL → HTML parse করে একটি Document |
| `DirectoryLoader` | Folder → glob pattern দিয়ে সব matching file |

### পরবর্তী ধাপ:
এই Document গুলো পরে **Text Splitter** দিয়ে chunk করা হয়, তারপর **Embedding** করে **Vector Store**-এ সংরক্ষণ করা হয় — এটাই RAG pipeline-এর মূল ভিত্তি।